# Concatenación y homogeneización de bases de isótopos foliares

Este notebook combina las 4 bases de datos (`Galeras_cellulose.xlsx`, `SV_cellulose.xlsx`, `Sumaco_cellulose.xlsx`, `iWUE_OYC_GUA.xlsx`) en una sola tabla homogeneizada, añade la **elevación por parcela** a partir de `elevation_dict.py`, y genera un **reporte de errores/inconsistencias** que deben revisarse manualmente.

**Qué hace el script, en orden:**
1. Carga cada archivo.
2. Estandariza el nombre de columnas clave (`Sample_ID`, `Plot`, `Site`, etc.).
3. Limpia el código de parcela (`Plot`): quita el prefijo `p`, ceros a la izquierda y el `.0` de los que venían como decimal.
4. Homogeneiza las columnas de isótopos que venían en dos unidades distintas (fracción 0–1 vs. porcentaje): `[C]%`, `[C_b]%`, `[N]`.
5. Concatena las 4 tablas en una sola.
5b. Añade los resultados bulk del laboratorio (2026): llena `[C_b]%`, `bulk_δ13C`, `[N]` y `δ15N` donde estaban vacíos y agrega como filas nuevas las muestras que no estaban en la base.
6. Añade la columna `Elevation_m` cruzando `(Site, Plot)` contra `elevation_dict`.
7. Corre 3 controles de calidad (QC) y marca en `QC_flag` las filas que necesitan revisión manual.
8. Guarda el resultado en un Excel con 3 hojas: `Datos_concatenados` (todo), `Errores_a_revisar` (solo las filas marcadas) y `Bulk_verificar` (cada ID del laboratorio con lo que se hizo con él).

## 1. Librerías

In [164]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)

## 2. Diccionario de elevación por parcela

Tomado de `elevation_dict.py`. Las llaves son tuplas `(Sitio, Parcela)`. **Importante:** este diccionario solo cubre los sitios `Galeras`, `Oyacachi` y `Sumaco`. Los sitios `SV` y `Guacamayos` (presentes en tus archivos `SV_cellulose.xlsx` e `iWUE_OYC_GUA.xlsx`) **no tienen elevación asignada todavía** — quedarán marcados en el reporte de errores para que agregues esos valores manualmente.

In [165]:
elevation_dict = {
    # Galeras
    ("Galeras", 20): 1090, ("Galeras", 21): 1080, ("Galeras", 22): 1060,
    ("Galeras", 23): 1090, ("Galeras", 24): 1110, ("Galeras", 29): 1050,
    ("Galeras", 47): 1000, ("Galeras", 48): 1130, ("Galeras", 49): 1080,
    ("Galeras", 28): 1597,
    ("Galeras", 25): 1450, ("Galeras", 26): 1560, ("Galeras", 50): 1557,
    ("Galeras", 60): 1570, ("Galeras", 61): 1560, ("Galeras", 62): 1590,
    # Oyacachi
    ("Oyacachi", 81): 4000, ("Oyacachi", 82): 4000, ("Oyacachi", 83): 4000,
    ("Oyacachi", 84): 3500, ("Oyacachi", 85): 3500, ("Oyacachi", 86): 3500,
    # Sumaco
    ("Sumaco", 9): 2000, ("Sumaco", 10): 2000, ("Sumaco", 11): 2000,
    ("Sumaco", 12): 2000, ("Sumaco", 13): 2000,
    ("Sumaco", 16): 1500, ("Sumaco", 18): 1500,
    #Guacamayos 
      ("Guacamayos", 6): 2000, ("Guacamayos", 7): 1940, ("Guacamayos", 8): 1995,
        ("Guacamayos", 55): 2000,("Guacamayos", 27): 1990,
    # Selva Viva
    ("Selva Viva", 68): 502, ("Selva Viva", 69): 508, ("Selva Viva", 70): 508,
    ("Selva Viva", 72): 453, ("Selva Viva", 74):470
}

## 3. Funciones auxiliares

- `clean_plot`: normaliza el código de parcela a entero (quita `p`, ceros a la izquierda, `.0`).
- `homogenize_pct`: detecta si un valor está en fracción (0–1) o en porcentaje, y homogeneiza todo a **porcentaje**. Regla: si `abs(valor) < 1` se asume fracción y se multiplica por 100; si ya es `>= 1` se asume que ya está en %. Esta regla se validó revisando duplicados del mismo `Sample_ID` con y sin homogeneizar (p.ej. `p22-550` con 0.474 y 47.2 para la misma muestra: son la misma medición en dos unidades distintas).

In [166]:
def clean_plot(v):
    """Normaliza el codigo de parcela a entero."""
    if pd.isna(v):
        return pd.NA
    s = str(v).strip().lower()
    if s.endswith('.0'):
        s = s[:-2]
    if s.startswith('p'):
        s = s[1:]
    s = s.lstrip('0')
    if s == '':
        s = '0'
    try:
        return int(s)
    except ValueError:
        return pd.NA

def homogenize_pct(series):
    """Pasa fracciones (abs<1) a porcentaje (x100). Deja igual lo que ya esta en %.
    Los negativos con abs>=1 no se tocan: son fisicamente imposibles y se marcan luego como error."""
    def fix(v):
        if pd.isna(v):
            return v
        if abs(v) < 1:
            return v * 100
        return v
    return series.apply(fix)

def implied_plot(sample_id):
    """Extrae la parcela implicita en el Sample_ID (ej. 'p09-230' -> 9) para cruzarla con la columna Plot."""
    if pd.isna(sample_id):
        return pd.NA
    m = re.match(r'^p0*([0-9]+)-', str(sample_id).strip().lower())
    return int(m.group(1)) if m else pd.NA

TYPE_MAP = {'herbarium': 'leaf herbarium'}  # unifica nomenclatura de 'Type'

## 4. Cargar y estandarizar cada base

En esta sección se define, para cada archivo:
- `Site` (sitio de muestreo), tomado del nombre del archivo, o de la columna `Site` si ya existía (caso `iWUE_OYC_GUA.xlsx`, que trae `Oyacachi` y `Guacamayos`).
- Limpieza de `Plot` y de las columnas de isótopos/porcentaje.
- Columnas redundantes o duplicadas que se eliminan (con la razón indicada en el comentario).

### 4.1 Galeras

In [167]:
g = pd.read_excel(r"Galeras/Galeras_cellulose.xlsx")

g['Site'] = 'Galeras'
g['Plot'] = g['Plot'].apply(clean_plot)
for c in ['[C]%', '[C_b]%', '[N]']:
    g[c] = homogenize_pct(g[c])
g['Type'] = g['Type'].replace(TYPE_MAP)
g['source_file'] = 'Galeras_cellulose.xlsx'
g = g.rename(columns={'Sample-ID': 'Sample_ID'})
g.head(2)

,Unnamed: 0,Sample_ID,[C]%,δ13C (‰ v.s.V-PDB),Type,[N],[C_b]%,δ15N (‰ v.s. V-PDB),bulk_δ13C (‰ v.s.V-PDB),Year,old_treeID,description,Type 2,Plot,new_TreeID,family,JH herbarium collections,genus,species,Site,source_file
0,0,p23-557,45.552995,-29.289590,leaf pulverized,1.912373,48.553106,1.960678,-31.458731,2006,557.0,NaN,NaN,23,NaN,Quiinaceae,3737,Lacunaria,crenata,Galeras,Galeras_cellulose.xlsx
1,1,p23-557,43.360479,-29.427788,leaf pulverized,1.912373,48.553106,1.960678,-31.458731,2006,557.0,NaN,NaN,23,NaN,Quiinaceae,3737,Lacunaria,crenata,Galeras,Galeras_cellulose.xlsx


### 4.2 SV

In [168]:
sv = pd.read_excel(r"Selva Viva\SV_cellulose.xlsx")

sv['Site'] = 'SV'
sv['Plot'] = sv['Plot'].apply(clean_plot)
for c in ['[C]%', '[C_b]%', '[N]']:
    sv[c] = homogenize_pct(sv[c])
sv['Type'] = sv['Type'].replace(TYPE_MAP)
sv['source_file'] = 'SV_cellulose.xlsx'
sv = sv.rename(columns={'Sample-ID': 'Sample_ID'})
sv["Site"] = "Selva Viva"
if 'sample-ID' in sv.columns:            # columna duplicada (minuscula), redundante con Sample_ID
    sv = sv.drop(columns=['sample-ID'])
sv.head(2)

,Unnamed: 0,Sample_ID,[C]%,δ13C (‰ v.s.V-PDB),Year,Plot,Type,numeroCampo,description,old_treeID,[N],[C_b]%,δ15N (‰ v.s. V-PDB),bulk_δ13C (‰ v.s.V-PDB),new_TreeID,family,JH herbarium collections,genus,species,Site,source_file
0,110,jh-p70-3904,39.990000,-31.980000,2006,70,leaf herbarium,3904.0,plot 70 species 2,NaN,NaN,NaN,NaN,NaN,3604.0,Meliaceae,NaN,Guarea,NaN,Selva Viva,SV_cellulose.xlsx
1,14,jh-3939,43.988841,-32.233186,2006,68,leaf herbarium,3939.0,plot 68 species 15,NaN,NaN,NaN,NaN,NaN,3604.0,Meliaceae,NaN,Guarea,NaN,Selva Viva,SV_cellulose.xlsx


### 4.3 Sumaco

In [169]:
su = pd.read_excel((r"Sumaco\Sumaco_cellulose.xlsx"))
su['Site'] = 'Sumaco'
su['Plot'] = su['Plot'].apply(clean_plot)
su['[C]%'] = homogenize_pct(su['[C]%'])   # este archivo no trae [C_b]% ni [N]
su['source_file'] = 'Sumaco_cellulose.xlsx'
su = su.rename(columns={'Sample-ID': 'Sample_ID'})
su.head(2)

,Unnamed: 0,Sample_ID,family,genus,species,[C]%,δ13C (‰ v.s.V-PDB),Plot,Year,old_treeID,new_TreeID,JH herbarium collections,Type,Site,source_file
0,0,p09-225,Moraceae,Ficus,quijosana,42.578188,-28.691043,9,2006,225,NaN,NaN,leaf pulverized,Sumaco,Sumaco_cellulose.xlsx
1,1,p09-230,Lauraceae,Ocotea,insularis,45.456318,-29.289822,1,2006,230,NaN,3343.0,leaf pulverized,Sumaco,Sumaco_cellulose.xlsx


### 4.4 iWUE (Oyacachi / Guacamayos)

In [170]:
iw = pd.read_excel(r"Oyacachi-Guacamayos\iWUE_OYC_GUA.xlsx")
iw['Plot'] = iw['Plot'].apply(clean_plot)
for c in ['[C]%', '[C_b]%', '[N]']:
    iw[c] = homogenize_pct(iw[c])
iw['Type'] = iw['Type'].replace(TYPE_MAP)
iw['source_file'] = 'iWUE_OYC_GUA.xlsx'
iw = iw.rename(columns={'Sample-ID': 'Sample_ID',
                         'species2': 'species',
                         'herbarium specimen': 'JH herbarium collections'})
if '[C]%_new' in iw.columns:   # es un duplicado exacto de [C]% donde existe -> se elimina
    iw = iw.drop(columns=['[C]%_new'])
for c in ['species3', 'species4', 'species5']:  # desgloses finos (cf/aff), poco poblados -> se descartan
    if c in iw.columns:
        iw = iw.drop(columns=[c])
iw.head(2)

,Unnamed: 0,new_TreeID,old_treeID,Plot,Site,family,leaf_sample_ID_2011,JH herbarium collections,[C]%,δ13C (‰ v.s.V-PDB),[N],δ15N (‰ v.s. V-PDB),[C_b]%,bulk_δ13C (‰ v.s.V-PDB),Year,Type,Sample_ID,genus_species,genus,species,comment,source_file
0,0,4039.0,4039.0,82,Oyacachi,Solanaceae,56077.0,4194.0,45.059988,-24.65,1.518827,-2.154079,44.042720,-27.419388,2011,leaf pulverized,p82-4039,Solanum stenophyllum,Solanum,stenophyllum,NaN,iWUE_OYC_GUA.xlsx
1,1,8016.0,4020.0,81,Oyacachi,Rosaceae,56069.0,NaN,45.781855,-26.37,2.984893,0.134936,45.888802,-28.960106,2011,leaf pulverized,p81-4020,Polylepis pauta,Polylepis,pauta,NaN,iWUE_OYC_GUA.xlsx


## 5. Concatenar las 4 bases

Se concatenan por nombre de columna (`sort=False` conserva el orden), rellenando con `NaN` las columnas que no existen en todas las bases (por ejemplo `PlotID` solo existe en Sumaco).

In [171]:
all_df = pd.concat([g, sv, su, iw], ignore_index=True, sort=False)
all_df = all_df.drop(columns=['Unnamed: 0'], errors='ignore')
print('Filas totales:', len(all_df))
all_df['Site'].value_counts(dropna=False)


# Diccionario con las correcciones
plot_fix = {
    "p28-8878": 28,
    "p60-8885": 60,
    "p48-3698": 48,
    "p68-8878": 68,
    "p68-8885": 68,
    "p10-252": 10,
    "p12-304": 12,
    "p13-315": 13,
    "p09-230": 9, #decia 1 en lugar de 9,
    "p62-8669":48, #no coincide el plot pero verificado manualmente
    "p28-8974":23, # 

}

# Suponiendo que la columna con esos códigos se llama "Sample"
for sample, plot in plot_fix.items():
    all_df.loc[all_df["Sample_ID"] == sample, "Plot"] = plot

# (Opcional) dejar Plot como entero
##df["Plot"] = df["Plot"].astype("Int64")
# Muestras afectadas

# Columnas a intercambiar
col1 = "[C_b]%"
col2 = "δ15N (‰ v.s. V-PDB)"
# p82-4044 únicamente para 2011
mask1 = (all_df["Sample_ID"] == "p82-4044") & (all_df["Year"] == 2011)
mask2 = all_df["Sample_ID"] == "p81-8023"

# Intercambiar los valores
for mask in [mask1, mask2]:
    all_df.loc[mask, [col1, col2]] = all_df.loc[mask, [col2, col1]].to_numpy()


Filas totales: 669


## 5b. Añadir resultados bulk del laboratorio (C/N, δ13C, δ15N — 2026)

Archivo: `260422specs - send to Andrea_20260928.xlsx` (hoja `Resultaten`, 5 secuencias: 260325, 260422, 260603, 260604, 260605).

**Qué hace esta sección:**
1. Lee la hoja `Resultaten`. `[C]` y `[N]` vienen como **fracción** (0.44) → se pasan a **%** (44 %). Los comentarios de texto que el laboratorio escribió dentro de columnas numéricas (`no peak`, `double peak`, `dil?? for C`) se mueven a `lab_comment` y el valor queda `NaN`.
2. Descarta las filas sin resultado (toda la **secuencia 4** — la hoja `Specs` dice *"rerun needed for the complete run"* — y 14 muestras de la secuencia 5). No se añaden a la base; quedan listadas para re-medir.
3. Las muestras medidas 2 veces se **promedian** (`bulk26_n_rep` = 2) y se marca si las réplicas difieren > 1 ‰. Un mismo voucher escrito como `jh-3842` y `jh-p69-3842` se trata como la misma muestra.
4. Cruza cada ID del laboratorio con la base, en este orden:
   - **ID exacto** normalizado (minúsculas, `_`→`-`, sin `*`, `jh-24-`→`jh-p24-`, `69-8893`→`p69-8893`).
   - **Nº de herbario + parcela**: `jh-p69-3855` ↔ `jh-3855` (solo si la parcela coincide con la de la base).
   - **Sufijo `-006`**: `p11-266-006` ↔ `p11-266` del año 2006.
   - **`leaf_sample_ID_2011`**: IDs de 5 dígitos (`56074`) ↔ Oyacachi 2011.
5. Si hay match: los valores se escriben en las columnas `bulk26_*` y **también** en las columnas estándar (`[C_b]%`, `bulk_δ13C`, `[N]`, `δ15N`) **solo si estaban vacías**. Nunca se sobreescribe un valor original de la base (la columna `bulk_source` indica de dónde viene cada valor).
6. Si no hay match: se añade como **fila nueva** (`source_file = 'bulk_lab_2026'`), con `Site` inferido de la parcela del ID. Si existe otra fila del mismo árbol/voucher (por `new_TreeID`, `old_treeID` o `JH herbarium collections`), se copia la taxonomía y se marca para verificar. `Year` queda vacío para que lo completes.
7. Genera la tabla `bulk_review`, que se guarda como hoja `Bulk_verificar` en el Excel final.

> Nota de unidades: el laboratorio reporta δ15N **vs. AIR** (la referencia correcta para N). El nombre de columna `δ15N (‰ v.s. V-PDB)` de la base es heredado; conviene renombrarlo en tus scripts por sitio.

In [ ]:
BULK_PATH = r"bulk\260422specs - send to Andrea_20260928.xlsx"   # <- ajusta la ruta a donde guardaste el archivo del laboratorio

def load_bulk(path):
    """Lee la hoja 'Resultaten' del laboratorio y la deja en formato largo y limpio."""
    raw = pd.read_excel(path, sheet_name='Resultaten', header=None, skiprows=2).iloc[:, :12]
    raw.columns = ['ID_lab', 'N', 'N_sd', 'extra', 'C', 'C_sd', 'd15N', 'd15N_sd',
                   'd13C', 'd13C_sd', 'Batch', 'ID_lab2']
    raw = raw[raw['ID_lab'].notna()].copy()
    raw['ID_lab'] = raw['ID_lab'].astype(str).str.strip()
    raw['Sequence'] = raw['Batch'].str.extract(r'[Ss]equentie\s*(\d+)')[0].astype('Int64')
    # El laboratorio escribio comentarios de texto dentro de columnas numericas ('no peak', 'dil?? for C', 'double peak')
    num_cols = ['N', 'N_sd', 'C', 'C_sd', 'd15N', 'd15N_sd', 'd13C', 'd13C_sd']
    raw['lab_comment'] = ''
    for c in num_cols:
        txt = raw[c].notna() & pd.to_numeric(raw[c], errors='coerce').isna()
        raw.loc[txt, 'lab_comment'] += c + ': ' + raw.loc[txt, c].astype(str) + '; '
        raw[c] = pd.to_numeric(raw[c], errors='coerce')
    # [C] y [N] vienen como fraccion (0.44 = 44 %) -> se pasan a % como el resto de la base
    for c in ['N', 'N_sd', 'C', 'C_sd']:
        raw[c] = raw[c] * 100
    raw['has_result'] = raw[['N', 'C', 'd15N', 'd13C']].notna().any(axis=1)
    return raw.drop(columns=['extra', 'ID_lab2'])

def norm_id(s):
    """Normaliza un Sample_ID para compararlo: minusculas, '_'->'-', sin '*', 'jh-24-' -> 'jh-p24-', '69-8893' -> 'p69-8893', p9 -> p09."""
    if pd.isna(s):
        return pd.NA
    s = str(s).strip().lower().replace('_', '-').replace('*', '')
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'^jh-(\d+)-(\d+)$', r'jh-p\1-\2', s)
    s = re.sub(r'^(\d{1,3})-(\d+)$', r'p\1-\2', s)
    s = re.sub(r'^(jh-)?p0*(\d+)-', lambda m: (m.group(1) or '') + 'p%02d-' % int(m.group(2)), s)
    return s

def parse_id(s):
    """(tipo, parcela, numero). tipo: 'jh' (herbario), 'p' (arbol), 'p_006' (arbol + sufijo -006), 'leaf2011', 'otro'."""
    s = norm_id(s)
    if pd.isna(s):
        return 'otro', pd.NA, pd.NA
    m = re.match(r'^jh-(?:p(\w+)-)?(\d+)$', s)          # jh-3855, jh-p69-3855, jh-pxx-3708, jh-px8-3712
    if m:
        plot = int(m.group(1)) if m.group(1) and m.group(1).isdigit() else pd.NA
        return 'jh', plot, int(m.group(2))
    m = re.match(r'^p(\d+)-(\d+)-006$', s)
    if m: return 'p_006', int(m.group(1)), int(m.group(2))
    m = re.match(r'^p(\d+)-(\d+)$', s)
    if m: return 'p', int(m.group(1)), int(m.group(2))
    m = re.match(r'^(\d{5})$', s)
    if m: return 'leaf2011', pd.NA, int(m.group(1))
    return 'otro', pd.NA, pd.NA


### 5b.1 Cargar resultados y promediar réplicas

In [ ]:
bulk_raw = load_bulk(BULK_PATH)
bulk_raw['nid'] = bulk_raw['ID_lab'].apply(norm_id)
# Un mismo voucher a veces viene como 'jh-3842' y otras como 'jh-p69-3842': se unifican al ID con parcela
_jh = bulk_raw['nid'].apply(parse_id)
_withplot = {}
for nid, (k, plot, num) in zip(bulk_raw['nid'], _jh):
    if k == 'jh' and pd.notna(plot):
        _withplot.setdefault(num, set()).add(nid)
for i, (k, plot, num) in _jh.items():
    if k == 'jh' and pd.isna(plot) and len(_withplot.get(num, ())) == 1:
        bulk_raw.at[i, 'nid'] = next(iter(_withplot[num]))
print('Filas en el archivo del laboratorio:', len(bulk_raw))
print('  con resultado:', bulk_raw['has_result'].sum(), '| sin resultado (no peak / secuencia 4 / pendientes):', (~bulk_raw['has_result']).sum())

# Muestras medidas 2 veces (secuencias 3 y 5): se promedian y se guarda n y el rango para revisar
bulk_ok = bulk_raw[bulk_raw['has_result']]
agg = bulk_ok.groupby('nid').agg(
    ID_lab=('ID_lab', lambda s: ' | '.join(s.unique())),
    Sequence=('Sequence', lambda s: ','.join(map(str, sorted(s.unique())))),
    C=('C', 'mean'), N=('N', 'mean'), d13C=('d13C', 'mean'), d15N=('d15N', 'mean'),
    n_rep=('ID_lab', 'size'),
    dif_d13C=('d13C', lambda s: s.max() - s.min()),
    dif_d15N=('d15N', lambda s: s.max() - s.min()),
    lab_comment=('lab_comment', lambda s: ''.join(s)),
).reset_index()
agg.loc[agg['n_rep'] == 1, ['dif_d13C', 'dif_d15N']] = np.nan
print('Muestras unicas con resultado:', len(agg), '| medidas 2 veces (promediadas):', (agg['n_rep'] > 1).sum())


### 5b.2 Cruzar IDs del laboratorio con la base

In [ ]:
PLOT_SITE = {p: s for (s, p) in elevation_dict.keys()}
PLOT_SITE.update({int(p): s for s, p in all_df[['Site', 'Plot']].dropna().drop_duplicates().itertuples(index=False)})

base = all_df
base['_nid'] = base['Sample_ID'].apply(norm_id)
_parsed = base['Sample_ID'].apply(parse_id)
base['_kind'] = _parsed.str[0]
base['_num'] = _parsed.str[2]
base['_newtree'] = pd.to_numeric(base['new_TreeID'], errors='coerce')
base['_oldtree'] = pd.to_numeric(base['old_treeID'], errors='coerce')
base['_leaf2011'] = pd.to_numeric(base['leaf_sample_ID_2011'], errors='coerce')

def find_matches(nid):
    """Filas de la base que son LA MISMA muestra fisica. Devuelve (indices, metodo, nota)."""
    idx = base.index[base['_nid'] == nid]
    if len(idx):
        return list(idx), 'ID exacto', ''
    kind, plot, num = parse_id(nid)
    if kind == 'jh':
        cand = base.index[(base['_kind'] == 'jh') & (base['_num'] == num)]
        if len(cand):
            plots = set(base.loc[cand, 'Plot'].dropna())
            if pd.isna(plot):
                return list(cand), 'n° herbario (parcela ilegible en ID lab)', f'Parcela en base: {sorted(plots)}'
            if plot in plots:
                return list(cand), 'n° herbario + parcela', ''
            return [], 'sin match', f'n° herbario {num} existe en la base pero en parcela {sorted(plots)}, no {plot}'
    if kind == 'p_006':
        cand = base.index[(base['_nid'] == f'p{plot:02d}-{num}') & (base['Year'] == 2006)]
        if len(cand):
            return list(cand), 'sufijo -006 retirado', ''
    if kind == 'leaf2011':
        cand = base.index[base['_leaf2011'] == num]
        if len(cand):
            return list(cand), 'leaf_sample_ID_2011', ''
    return [], 'sin match', ''

def related_rows(nid):
    """Para muestras SIN match: filas de la base del mismo arbol/voucher (solo para heredar taxonomia y dar pistas)."""
    kind, plot, num = parse_id(nid)
    if pd.isna(num):
        return []
    same_plot = (base['Plot'] == plot) if pd.notna(plot) else True
    for col in ['_newtree', '_oldtree']:
        cand = base.index[(base[col] == num) & same_plot]
        if len(cand):
            return list(cand)
    jh = base['JH herbarium collections'].astype(str).str.contains(rf'(?<!\d){num}(?!\d)', regex=True, na=False)
    cand = base.index[jh & same_plot]
    return list(cand)

matches = []
for r in agg.itertuples():
    idx, method, note = find_matches(r.nid)
    matches.append({'nid': r.nid, 'idx': idx, 'match_method': method, 'match_note': note})
matches = pd.DataFrame(matches)
agg = agg.merge(matches, on='nid')

# Una fila de la base no puede recibir datos de 2 IDs distintos del laboratorio -> conflicto, no se llena
row_owner = agg.explode('idx').dropna(subset=['idx']).groupby('idx')['nid'].nunique()
conflict_rows = set(row_owner[row_owner > 1].index)
agg['conflicto'] = agg['idx'].apply(lambda ix: any(i in conflict_rows for i in ix))
print(agg['match_method'].value_counts().to_string())
print('Muestras en conflicto (2 IDs lab -> misma fila):', agg['conflicto'].sum())


### 5b.3 Llenar columnas bulk y añadir filas nuevas

In [ ]:
BULK_MAP = {'C': '[C_b]%', 'd13C': 'bulk_δ13C (‰ v.s.V-PDB)', 'N': '[N]', 'd15N': 'δ15N (‰ v.s. V-PDB)'}
NEW_COLS = {'C': 'bulk26_[C]%', 'd13C': 'bulk26_δ13C', 'N': 'bulk26_[N]', 'd15N': 'bulk26_δ15N'}
for c in list(NEW_COLS.values()) + ['bulk26_ID_lab', 'bulk26_Sequence', 'bulk26_n_rep', 'bulk26_match', 'bulk26_flag']:
    all_df[c] = np.nan if not c.endswith(('ID_lab', 'Sequence', 'match', 'flag')) else ''
all_df['bulk_source'] = np.where(all_df['[C_b]%'].notna() | all_df['δ15N (‰ v.s. V-PDB)'].notna(), 'base original', '')

def _range_flags(r):
    f = []
    if pd.notna(r.C) and not (35 <= r.C <= 55): f.append(f'[C] fuera de rango ({r.C:.1f}%)')
    if pd.notna(r.N) and not (0.5 <= r.N <= 5): f.append(f'[N] fuera de rango ({r.N:.2f}%)')
    if pd.notna(r.d13C) and not (-38 <= r.d13C <= -24): f.append('δ13C fuera de rango')
    if pd.notna(r.d15N) and not (-8 <= r.d15N <= 10): f.append('δ15N fuera de rango')
    if r.n_rep > 1 and ((r.dif_d13C > 1) or (r.dif_d15N > 1)): f.append('replicas difieren >1‰')
    if r.lab_comment: f.append('comentario lab: ' + r.lab_comment.strip('; '))
    return f

new_rows = []
for c in ['accion', 'flags', 'Site', 'Sample_ID_base']:
    agg[c] = pd.Series('', index=agg.index, dtype=object)
for r in agg.itertuples():
    flags = _range_flags(r)
    if r.match_note: flags.append(r.match_note)
    vals = {NEW_COLS[k]: getattr(r, k) for k in NEW_COLS}
    info = {'bulk26_ID_lab': r.ID_lab, 'bulk26_Sequence': r.Sequence, 'bulk26_n_rep': r.n_rep, 'bulk26_match': r.match_method}
    if r.idx and not r.conflicto:
        ix = r.idx
        # siempre se guardan los valores nuevos en las columnas bulk26_*
        for k, v in {**vals, **info}.items():
            all_df.loc[ix, k] = v
        # se copian a las columnas estandar SOLO si estaban vacias (nunca se sobreescribe la base original)
        had = all_df.loc[ix, list(BULK_MAP.values())].notna().any(axis=None)
        if had:
            flags.append('la base ya tenia valores bulk: NO se sobreescribieron, comparar con bulk26_*')
            agg.loc[r.Index, 'accion'] = 'match, base ya tenia bulk (solo bulk26_*)'
        else:
            for k, col in BULK_MAP.items():
                all_df.loc[ix, col] = getattr(r, k)
            all_df.loc[ix, 'bulk_source'] = f'lab 2026 (Seq {r.Sequence})'
            agg.loc[r.Index, 'accion'] = 'match, columnas bulk llenadas'
        agg.loc[r.Index, 'Site'] = all_df.loc[ix[0], 'Site']
        agg.loc[r.Index, 'Sample_ID_base'] = ', '.join(all_df.loc[ix, 'Sample_ID'].astype(str).unique())
        all_df.loc[ix, 'bulk26_flag'] = '; '.join(flags)
    else:
        # Sin match (o en conflicto): se anade como fila nueva, con Site inferido de la parcela
        kind, plot, num = parse_id(r.nid)
        if r.conflicto:
            flags.append('CONFLICTO: otra muestra del lab cae en la misma fila de la base')
        rel = related_rows(r.nid)
        row = {'Sample_ID': r.ID_lab, 'Plot': plot if pd.notna(plot) else pd.NA,
               'Site': PLOT_SITE.get(plot, np.nan) if pd.notna(plot) else np.nan,
               'Type': 'leaf herbarium' if kind == 'jh' else np.nan,
               'source_file': 'bulk_lab_2026', 'bulk_source': f'lab 2026 (Seq {r.Sequence})',
               **{col: getattr(r, k) for k, col in BULK_MAP.items()}, **vals, **info}
        if rel:
            src = all_df.loc[rel[0]]
            for c in ['family', 'genus', 'species', 'old_treeID', 'new_TreeID', 'JH herbarium collections']:
                row[c] = src.get(c)
            if pd.isna(row['Site']):
                row['Site'] = src['Site']
            flags.append(f'taxonomia copiada de {src["Sample_ID"]} (mismo arbol/voucher): verificar')
        if pd.isna(row['Site']):
            flags.append('Site desconocido: parcela no interpretable en el ID')
        flags.append('fila nueva (sin cellulosa en la base); Year por completar')
        row['bulk26_flag'] = '; '.join(flags)
        new_rows.append(row)
        agg.loc[r.Index, 'Site'] = row['Site']
        agg.loc[r.Index, 'Sample_ID_base'] = ', '.join(all_df.loc[rel, 'Sample_ID'].astype(str).unique()) if rel else ''
        agg.loc[r.Index, 'accion'] = 'fila nueva'
    agg.loc[r.Index, 'flags'] = '; '.join(flags)

all_df = pd.concat([all_df, pd.DataFrame(new_rows)], ignore_index=True, sort=False)
all_df = all_df.drop(columns=[c for c in all_df.columns if c.startswith('_')])
print(agg['accion'].value_counts().to_string())
print('Filas totales tras anadir bulk:', len(all_df))

# Tabla de trabajo para revisar (se guarda como hoja aparte en el Excel final)
_no_result = bulk_raw[~bulk_raw['has_result'] & ~bulk_raw['nid'].isin(agg['nid'])][['ID_lab', 'Sequence', 'lab_comment']].copy()
_no_result['accion'] = 'sin resultado (no se anade): re-medir'
def _site_of(s):
    kind, plot, num = parse_id(s)
    if pd.notna(plot):
        return PLOT_SITE.get(plot, np.nan)
    ix = find_matches(norm_id(s))[0]
    return all_df.loc[ix[0], 'Site'] if ix else np.nan
_no_result['Site'] = _no_result['ID_lab'].apply(_site_of)
_no_result['flags'] = _no_result['lab_comment'].str.strip('; ')
bulk_review = pd.concat([
    agg[['Site', 'ID_lab', 'Sample_ID_base', 'Sequence', 'n_rep', 'C', 'N', 'd13C', 'd15N', 'dif_d13C', 'dif_d15N',
         'match_method', 'accion', 'flags']],
    _no_result.drop(columns='lab_comment')], ignore_index=True, sort=False)
bulk_review['Site'] = bulk_review['Site'].fillna('Sin sitio')
bulk_review = bulk_review.sort_values(['Site', 'accion', 'ID_lab']).reset_index(drop=True)


## 6. Añadir elevación por parcela

Se cruza `(Site, Plot)` contra `elevation_dict`. Donde no hay coincidencia, `Elevation_m` queda en `NaN`.

In [172]:
def get_elev(row):
    return elevation_dict.get((row['Site'], row['Plot']), np.nan)

all_df['Elevation_m'] = all_df.apply(get_elev, axis=1)
all_df['Elevation_m'].isna().sum()

0

## 7. Control de calidad (QC)

Se marca cada fila en la columna `QC_flag` con uno o más de estos motivos:

1. **`[C_b]% negativo`**: hay 2 filas (site Oyacachi) donde el carbono a granel homogeneizado da negativo — dato imposible, seguramente error de captura/instrumento. Revisar el dato crudo.
3. **`Plot vacio/no interpretable`**: la celda de parcela venía vacía o con un valor no interpretable como número.
4. **`Plot no coincide con prefijo del Sample_ID`**: el código de parcela en `Sample_ID` (ej. `p09-230` implica parcela 9) no coincide con el valor de la columna `Plot` de esa fila (ej. decía `01`). Son 3 filas — probablemente errores de tipeo en la planilla original.
5. **`Bulk 2026: ver bulk26_flag`**: filas que recibieron datos del laboratorio (sección 5b) con algo que revisar: valor fuera de rango, réplicas que difieren > 1 ‰, comentario del laboratorio, fila nueva sin match o taxonomía copiada de otro registro.


In [173]:
print(all_df[all_df["Type"].isna()]["Sample_ID"])#que filas les falta inofmracion, segun esto estas pueden ir com 
mask = all_df["Type"].isna() #creo una mascara con los que son nan
all_df.loc[mask, "Type"] = "leaf pulverized" #se añade leaf pulverized

38      p23-565
39      p23-578
40      p23-566
41      p29-792
392    p68-8878
398    p68-8885
658     p06-157
659     p27-669
660     p27-682
661     p27-687
662     p27-694
663     p27-695
664     p27-696
665     p27-704
666     p27-708
667     p27-712
668     p27-714
Name: Sample_ID, dtype: object


In [ ]:
all_df['QC_flag'] = ''

# 1. [C_b]% fisicamente imposible
neg_mask = all_df['[C_b]%'] < 0
all_df.loc[neg_mask, 'QC_flag'] += '[C_b]% negativo (revisar manualmente); '

# 2. Sin elevacion asignada
elev_missing_mask = all_df['Elevation_m'].isna() & all_df['Plot'].notna()
all_df.loc[elev_missing_mask, 'QC_flag'] += 'Sin elevacion en elevation_dict; '

# 3. Plot vacio
plot_missing_mask = all_df['Plot'].isna()
all_df.loc[plot_missing_mask, 'QC_flag'] += 'Plot vacio/no interpretable; '

# 4. Plot no coincide con el Sample_ID
all_df['Plot_from_ID'] = all_df['Sample_ID'].apply(implied_plot)
mismatch_mask = (all_df['Plot_from_ID'].notna()) & (all_df['Plot'].notna()) & (all_df['Plot_from_ID'] != all_df['Plot'])
all_df.loc[mismatch_mask, 'QC_flag'] += 'Plot no coincide con prefijo del Sample_ID; '

# 5. Resultados bulk 2026 que requieren revision (fuera de rango, replicas distintas, filas nuevas, taxonomia copiada...)
bulk_flag_mask = all_df['bulk26_flag'].fillna('').ne('')
all_df.loc[bulk_flag_mask, 'QC_flag'] += 'Bulk 2026: ver bulk26_flag; '

print('Filas marcadas:', (all_df['QC_flag'] != '').sum(), 'de', len(all_df))

## 8. Reordenar columnas y guardar resultados

Se guarda un Excel con dos hojas:
- `Datos_concatenados`: la tabla completa homogeneizada.
- `Errores_a_revisar`: solo las filas con `QC_flag` distinto de vacío, para que las revises manualmente.

In [ ]:
front_cols = ['Site', 'Plot', 'Elevation_m', 'Sample_ID', 'Year', 'Type',
              'family', 'genus', 'species',
              '[C]%', 'δ13C (‰ v.s.V-PDB)',
              '[C_b]%', 'bulk_δ13C (‰ v.s.V-PDB)',
              '[N]', 'δ15N (‰ v.s. V-PDB)', 'bulk_source',
              'QC_flag']
other_cols = [c for c in all_df.columns if c not in front_cols]
all_df = all_df[front_cols + other_cols]

out_path = 'isotopos_concatenado.xlsx'
report_rows = all_df[all_df['QC_flag'] != ''][
    ['Sample_ID','Site','Plot','Plot_from_ID','Elevation_m','[C]%','[C_b]%',
     'δ15N (‰ v.s. V-PDB)','QC_flag','bulk26_flag','source_file']
]

all_df = all_df.drop(columns={"genus_species","comment","Plot_from_ID","Type2"})
#en la columna comment solo hay un comentario de un reeemplzao de un sp de una Aniba coto, y Type2 era para ver la Miscelanea

with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    all_df.to_excel(writer, sheet_name='Datos_concatenados', index=False)
    report_rows.to_excel(writer, sheet_name='Errores_a_revisar', index=False)
    bulk_review.to_excel(writer, sheet_name='Bulk_verificar', index=False)   # IDs del laboratorio a revisar por sitio

print(f'Guardado: {out_path}')
report_rows

## 9. Resumen de lo que debes revisar manualmente

- **Corregido: Elevación faltante para `SV` y `Guacamayos`**: agrega esas parcelas a `elevation_dict` (313 filas de `SV` y 90 de `Guacamayos` quedaron sin elevación).
- **Corregido: 2 filas con `[C_b]%` negativo** (site Oyacachi, `p82-4044` y `p81-8023`), que además tienen δ15N fuera de rango (~42–43‰, cuando el resto de la base está entre -5 y +8‰). Revisar el dato crudo de laboratorio.
- **Corregido. 3 filas con parcela ambigua** (`Plot` no coincide con el prefijo del `Sample_ID`): `p62-8669`, `p28-8974`, `p09-230`.
- Un puñado de filas con `Plot` vacío que no se pudieron interpretar.
- **Bulk 2026 (sección 5b)**: revisar la hoja `Bulk_verificar`, ordenada por sitio. Pendientes principales: IDs sin sitio (`46xxx-my/al`, `tapirila`, `parkia-8634`, `pxx-*`, `jh-p42-*`), filas nuevas sin `Year`, y las muestras de la secuencia 4 que el laboratorio debe volver a medir.
